# 🎙️ YouTube → Transcript for Clipperino

Generates a `transcript.csv` ready to import into **[Clipperino](https://github.com/AlexDeveloperUwU/clipperino)**.

## How to use

1. Run **Cell 1** once to install dependencies. On first run, go to **Runtime → Restart session** after it finishes, then skip Cell 1 on future runs.
2. Run **Cell 2**. It will ask you:
   - **YouTube URL** — paste and hit enter
   - **Language** — type `en` or `es`, or just press enter for English
   - **Cookies** — if a `cookies.txt` already exists it asks to reuse it; otherwise asks if you want to upload one (needed if YouTube blocks the download)
3. Everything else runs automatically: download → normalize → transcribe → download CSV.

## Notes

- Backend is auto-selected: **faster-whisper (GPU)** if a GPU is available, **openai-whisper (CPU)** otherwise.
- Cookies must be in **Netscape format** — export from Chrome/Firefox with the [Get cookies.txt LOCALLY](https://chromewebstore.google.com/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc) extension while logged into YouTube.

In [ ]:
!pip install -q faster-whisper openai-whisper yt-dlp ipywidgets
!apt-get install -q -y ffmpeg
print("✅ Done. If this is your first run, go to Runtime → Restart session, then run Cell 2.")

In [ ]:
import os, csv, glob, subprocess
import torch
from datetime import timedelta
from google.colab import files

WORK_DIR  = "/home/alexdevuwu"
AUDIO_EXT = (".m4a", ".mp3", ".wav", ".flac", ".ogg", ".opus", ".aac", ".mp4", ".webm")
os.makedirs(WORK_DIR, exist_ok=True)

URL  = input("YouTube URL: ").strip()
LANG = input("Language [en]: ").strip() or "en"

DEFAULT_COOKIES = os.path.join(WORK_DIR, "cookies.txt")
if os.path.exists(DEFAULT_COOKIES):
    ans = input(f"cookies.txt found at {DEFAULT_COOKIES}. Use it? [Y/n]: ").strip().lower()
    COOKIES = DEFAULT_COOKIES if ans != "n" else ""
else:
    ans = input("Upload cookies.txt? [y/N]: ").strip().lower()
    if ans == "y":
        print("📂 Select your cookies.txt...")
        uploaded = files.upload()
        if uploaded:
            name = list(uploaded.keys())[0]
            COOKIES = os.path.join(WORK_DIR, name)
            with open(COOKIES, "wb") as f: f.write(uploaded[name])
            print(f"✅ Cookies saved: {COOKIES}")
        else:
            COOKIES = ""
    else:
        COOKIES = ""

BACKEND = "faster" if torch.cuda.is_available() else "openai"
print(f"\n▶  URL     : {URL}")
print(f"▶  Language: {LANG}")
print(f"▶  Backend : {BACKEND} ({'GPU' if BACKEND == 'faster' else 'CPU'})")
print(f"▶  Cookies : {COOKIES or 'none'}\n")

CONFIG = {"lang": LANG}

def fmt(s): return str(timedelta(seconds=int(s)))

print("📥 Fetching audio...")
out_tpl = os.path.join(WORK_DIR, "audio_downloaded.%(ext)s")
cmd = ["yt-dlp", "--no-playlist", "--js-runtimes", "node", "--remote-components", "ejs:github",
       "-f", "bestaudio", "--extract-audio", "--audio-format", "wav", "--audio-quality", "0",
       "-o", out_tpl]
if COOKIES and os.path.exists(COOKIES):
    cmd += ["--cookies", COOKIES]
cmd.append(URL)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError("yt-dlp failed.")
wav = os.path.join(WORK_DIR, "audio_downloaded.wav")
if not os.path.exists(wav):
    matches = glob.glob(os.path.join(WORK_DIR, "audio_downloaded.*"))
    if not matches: raise FileNotFoundError("yt-dlp produced no audio file.")
    wav = matches[0]
audio_path = wav
print(f"✅ Audio: {audio_path}")

print("🔧 Normalizing to 16 kHz mono...")
dst = os.path.join(WORK_DIR, os.path.splitext(os.path.basename(audio_path))[0] + "_16k.wav")
if audio_path != dst:
    subprocess.run(["ffmpeg", "-y", "-i", audio_path, "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", dst],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    audio_path = dst
print(f"✅ Normalized: {audio_path}")

print(f"🧠 Transcribing ({BACKEND}, {LANG})...")
if BACKEND == "faster":
    from faster_whisper import WhisperModel
    print("  ⏳ Loading model...")
    model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16",
                         num_workers=4, download_root=os.path.join(WORK_DIR, "models"))
    print("  ✅ Model loaded.")
    segs_iter, info = model.transcribe(
        audio_path, language=LANG, beam_size=5, temperature=0.0,
        vad_filter=True, vad_parameters=dict(min_silence_duration_ms=500),
        no_speech_threshold=0.6, word_timestamps=False, condition_on_previous_text=False,
    )
    print(f"  ⚡ Duration: {timedelta(seconds=int(info.duration))}")
    segments = []
    for s in segs_iter:
        seg = {"start": fmt(s.start), "end": fmt(s.end), "text": s.text.strip()}
        print(f"  [{seg['start']} → {seg['end']}] {seg['text']}", flush=True)
        segments.append(seg)
    torch.cuda.empty_cache()
else:
    import whisper
    print("  ⏳ Loading model...")
    model = whisper.load_model("turbo", device="cpu")
    print("  ✅ Model loaded.")
    result = model.transcribe(audio_path, language=LANG, fp16=False, verbose=True)
    segments = [{"start": fmt(s["start"]), "end": fmt(s["end"]), "text": s["text"].strip()} for s in result["segments"]]

print(f"✅ {len(segments)} segments.")

csv_path = os.path.join(WORK_DIR, "transcript.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Start", "End", "Transcript"])
    for s in segments:
        writer.writerow([s["start"], s["end"], s["text"]])
print(f"💾 Saved: {csv_path}")
files.download(csv_path)
print("🎉 Done! Import transcript.csv into Clipperino.")
